# Creating Cohorts of Songs (Rolling Stones Spotify Dataset)
This notebook performs exploratory data analysis (EDA), dimensionality reduction (PCA), cluster analysis (K-Means), and builds a content-based recommendation engine to group the Rolling Stones' songs into cohorts based on their musical characteristics (audio features).

### Project Objectives & Roadmap:
1. **Initial Data Inspection & Cleaning:** Inspect metadata, identify missing values, investigate duplicate song IDs and duplicate track titles across albums, and examine feature distributions.
2. **Exploratory Data Analysis (EDA) & Feature Engineering:** Recommend top albums based on popular song count, analyze audio feature distributions, and examine the correlation between audio features and popularity—specifically exploring how this correlation has evolved over the decades.
3. **Dimensionality Reduction (PCA):** Address the curse of dimensionality, project high-dimensional audio features onto Principal Components, analyze component loadings to understand the musical meaning of PC1 and PC2, and visualize songs in 2D space.
4. **Cluster Analysis:** Determine optimal cluster count using the Elbow Method (WCSS) and Silhouette Analysis, fit a K-Means model, and visualize the song cohorts along with their cluster centroids in PCA space.
5. **Cohort Profiling:** Interpret the musical identity of each cohort through feature means, heatmaps, standardized comparative profiles, and real song examples.
6. **Cohort-Based Recommendation Engine (Lesson 07 Integration):** Build a content-based recommendation function that retrieves similar tracks within a song's assigned cohort using Cosine Similarity, with automatic title deduplication and cold-start support.


In [ ]:
# Detect environment and setup file imports for Google Colab compatibility
import os
try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

if IN_COLAB:
    print('Running in Google Colab environment.')
    print("Please upload the dataset file 'rolling_stones_spotify.csv' when prompted below...")
    from google.colab import files
    uploaded = files.upload()
else:
    print('Running in local Python environment. Path defaults to raw data folder.')


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# Set seaborn theme and plot sizes
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Path resilience: checks local directory path first, then Colab upload folder
try:
    df = pd.read_csv('1736848608_rolling_stones_spotify/rolling_stones_spotify.csv', index_col=0)
except FileNotFoundError:
    df = pd.read_csv('rolling_stones_spotify.csv', index_col=0)

print(f'Dataset loaded successfully. Shape (rows, columns): {df.shape}')
df.head()


## Step 1: Initial Data Inspection & Data Cleaning

### Why do we perform this step?
* **Data Integrity:** Ensure our models are not trained on erroneous, corrupted, or missing data.
* **Duplicate Investigation:** In music streaming catalogs, duplicate entries exist at two levels:
  1. *Duplicate Spotify IDs:* Exact database duplicates where the same track record is replicated (must be removed).
  2. *Duplicate Song Titles across Albums:* The same composition re-appearing on Deluxe Editions, Live Concert recordings, Remasters, or Best-Of Compilations (e.g., *'Paint It, Black'* or *'Start Me Up'*). These have different audio feature measurements (live crowd noise, modern remaster compression) and represent distinct audio recordings.
* **Feature Separation:** Distinguish metadata identifiers (`name`, `album`, `release_date`, `id`, `uri`, `track_number`) from musical audio features used for distance calculations.
* **Scale Disparities:** Check min/max/outlier ranges to prepare for standardization.


In [ ]:
# 1. Metadata Info & Null Values
print('=== Metadata Info ===')
df.info()

print('\n=== Missing Values Check ===')
print(df.isna().sum())

# 2. Duplicate checks (Unique ID vs. Song Title)
duplicate_ids = df.duplicated(subset=['id']).sum()
duplicate_titles = df.duplicated(subset=['name']).sum()
print(f'\nDuplicate tracks by unique Spotify ID: {duplicate_ids}')
print(f'Duplicate tracks by Song Title across albums: {duplicate_titles} (out of {len(df)} total tracks)')

# 3. Audio features definition
audio_features = ['acousticness', 'danceability', 'energy', 'instrumentalness', 'liveness', 'loudness', 'speechiness', 'tempo', 'valence']

# 4. Summary Statistics of Audio Features
print('\n=== Summary Statistics: Audio Features ===')
display(df[audio_features + ['popularity', 'duration_ms']].describe().round(3))

# 5. Outlier & Distribution Boxplots
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()
for idx, col in enumerate(audio_features + ['popularity']):
    sns.boxplot(y=df[col], ax=axes[idx], color='skyblue')
    axes[idx].set_title(f'Distribution: {col}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


### Step 1 Observations:
* **Shape & Completeness:** The dataset contains $1,610$ rows and $17$ columns. There are **$0$ missing values** across the entire dataset.
* **Database Duplicates:** There are **$0$ duplicate records by unique Spotify `id`**, confirming that every row represents an individual track recording.
* **Catalog Re-issues & Live Duplicates:** There are **$656$ duplicate song titles** (`name`). This occurs because the Rolling Stones catalog on Spotify includes original studio releases, 2009/2010 remasters, Deluxe bonus tracks, and live stadium albums (e.g., *Licked Live In NYC*, *Live At The El Mocambo*). Each recording has distinct audio features (e.g., live versions have high `liveness` $\ge 0.8$, whereas studio masters have higher loudness and lower acousticness). We retain all $1,610$ recordings for clustering, and implement title deduplication when building the recommendation engine.
* **Scale Variance:** Audio features like `acousticness`, `danceability`, `energy`, `valence`, `liveness`, and `speechiness` are bounded within $[0.0, 1.0]$. However, `loudness` ranges from $-24.4$ dB to $-2.1$ dB, and `tempo` ranges from $65$ to $216$ BPM. This demonstrates that **Standardization (`StandardScaler`) is strictly necessary** before clustering so that distance metrics are not dominated by `tempo` or `loudness`.


## Step 2: Exploratory Data Analysis & Feature Engineering

### 2.1 Recommending Albums based on Popular Songs
To identify the top two albums to recommend to general listeners:
1. Determine the $75$th percentile of track popularity.
2. Filter tracks meeting or exceeding this threshold.
3. Count the number of popular songs per album and select the top two.


In [ ]:
# Calculate the 75th percentile of popularity
popularity_75th = df['popularity'].quantile(0.75)
print(f'75th Percentile Popularity Threshold: {popularity_75th:.1f}')

# Filter tracks at or above the 75th percentile
popular_tracks = df[df['popularity'] >= popularity_75th]
album_popular_counts = popular_tracks.groupby('album')['name'].count().sort_values(ascending=False)

# Plot top 10 albums by popular songs count
plt.figure(figsize=(12, 6))
top_10_albums = album_popular_counts.head(10)
sns.barplot(x=top_10_albums.values, y=top_10_albums.index, hue=top_10_albums.index, palette='viridis', legend=False)
plt.title(f'Top 10 Albums by Count of Popular Songs (Popularity >= {popularity_75th:.1f})', fontsize=14, fontweight='bold')
plt.xlabel('Count of Popular Tracks', fontsize=12)
plt.ylabel('Album Name', fontsize=12)
plt.tight_layout()
plt.show()

print('=== Top 5 Recommended Albums by Popular Songs Count ===')
for rank, (album, count) in enumerate(album_popular_counts.head(5).items(), 1):
    print(f'{rank}. {album} ({count} popular tracks)')


### Step 2.1 Observations & Album Recommendations:
* **Popularity Threshold:** The $75$th percentile threshold is **$27.0$** (out of 100). Songs with scores $\ge 27$ represent the top quartile of the Rolling Stones catalog on Spotify.
* **Top Albums Identified:**
  1. **Honk (Deluxe)**: Contains **$18$ popular tracks**. *Honk* is an extensive greatest hits collection spanning six decades of music, making it the ultimate gateway for general listeners.
  2. **Exile On Main Street (2010 Re-Mastered)**: Contains **$18$ popular tracks**. Widely acclaimed as the band's magnum opus double album.
  3. *(Note on Deluxe editions)*: *Exile On Main Street (Deluxe Version)* also contains $18$ popular tracks, representing an alternative release of the same core album.
* **Final Album Recommendation:** We recommend **Honk (Deluxe)** (for comprehensive greatest-hits appeal) and **Exile On Main Street (2010 Re-Mastered)** (for the definitive classic studio album experience).


### 2.2 Audio Feature Distributions & Sonic Characteristics
We examine the distribution shapes (histograms and KDE curves) of the 9 core audio features to uncover the band's musical signature.


In [ ]:
# Plotting distributions for all 9 audio features
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for idx, feat in enumerate(audio_features):
    sns.histplot(df[feat], kde=True, ax=axes[idx], color='teal', bins=25)
    axes[idx].set_title(f'{feat.capitalize()} Distribution', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel(feat)
    axes[idx].set_ylabel('Count')

plt.suptitle('Audio Feature Distributions across Rolling Stones Catalog', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


### Step 2.2 Observations on Audio Feature Distributions:
* **High Energy & Loudness:** `energy` is heavily skewed toward high values (mean $\approx 0.79$), and `loudness` concentrates between $-4$ and $-8$ dB, reflecting the band's driving rock-and-roll sound.
* **Bimodal Liveness:** `liveness` exhibits a clear bimodal distribution with one spike below $0.3$ (studio recordings) and another major cluster above $0.7$ (live concert tracks), confirming the presence of distinct recording formats.
* **Positivity (Valence):** `valence` has a high average (mean $\approx 0.58$), showing that the Rolling Stones catalog is predominantly upbeat, groove-oriented, and celebratory blues-rock.
* **Acousticness & Speechiness:** `speechiness` is uniformly low (< 0.15), indicating melodic music rather than spoken word. `acousticness` is generally low but features a notable tail of acoustic ballads.


### 2.3 Correlation Analysis: Popularity vs. Audio Features
We compute the Pearson correlation coefficient ($r$) between song popularity and audio features to determine which attributes correlate with higher streaming numbers.


In [ ]:
# Compute Pearson correlation matrix
numeric_cols = audio_features + ['popularity', 'duration_ms']
corr_matrix = df[numeric_cols].corr()

# Plot Seaborn correlation heatmap
plt.figure(figsize=(11, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.3f', linewidths=0.5, vmin=-1, vmax=1)
plt.title('Correlation Matrix of Audio Features, Duration, and Popularity', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('=== Correlation with Popularity (Sorted) ===')
display(corr_matrix['popularity'].sort_values(ascending=False).to_frame(name='Pearson Correlation'))


### Step 2.3 Observations:
* **Positive Correlates:** `loudness` ($+0.156$) and `danceability` ($+0.141$) display the strongest positive correlations with popularity. Modern listeners favor punchier, rhythmically engaging studio tracks.
* **Negative Correlates:** `liveness` ($-0.206$) and `speechiness` ($-0.137$) have the strongest negative correlations with popularity. Casual listeners stream studio recordings far more frequently than audience-heavy live bootlegs.
* **Acousticness:** Exhibits a mild positive correlation ($+0.108$) overall, driven by enduring acoustic ballad favorites like *'Angie'* and *'Wild Horses'*.


### 2.4 Temporal Evolution of Feature Correlations with Popularity
The problem statement asks to: **"Examine the relationship between a song's popularity and various factors, exploring how this correlation has evolved."**

To answer this question, we extract the release year, group songs by decade (1960s to 2020s), and compute the correlation between popularity and each audio feature within each decade.


In [ ]:
# Extract release year and decade
df['release_year'] = pd.to_datetime(df['release_date'], errors='coerce').dt.year
df['decade'] = (df['release_year'] // 10) * 10

# Compute correlation of audio features with popularity for each decade
decade_corrs = {}
for decade, grp in df.groupby('decade'):
    if len(grp) >= 20:
        decade_corrs[f'{int(decade)}s'] = grp[audio_features + ['popularity']].corr()['popularity'].drop('popularity')

corr_evolution_df = pd.DataFrame(decade_corrs)

# Plotting the evolution heatmap
plt.figure(figsize=(12, 7))
sns.heatmap(corr_evolution_df, annot=True, cmap='coolwarm', fmt='.3f', linewidths=0.5, vmin=-0.5, vmax=0.5)
plt.title('Evolution of Feature Correlations with Popularity Across Decades', fontsize=14, fontweight='bold')
plt.xlabel('Decade of Release', fontsize=12)
plt.ylabel('Audio Feature', fontsize=12)
plt.tight_layout()
plt.show()

# Yearly trends of key features
yearly_trends = df.groupby('release_year')[['popularity', 'energy', 'acousticness', 'loudness']].mean().reset_index()

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.set_xlabel('Release Year', fontsize=12)
ax1.set_ylabel('Average Popularity', color='tab:blue', fontsize=12)
sns.lineplot(data=yearly_trends, x='release_year', y='popularity', ax=ax1, color='tab:blue', label='Popularity', marker='o')
ax1.tick_params(axis='y', labelcolor='tab:blue')

ax2 = ax1.twinx()
ax2.set_ylabel('Audio Metrics (Energy & Acousticness)', color='tab:red', fontsize=12)
sns.lineplot(data=yearly_trends, x='release_year', y='energy', ax=ax2, color='tab:red', label='Energy', marker='s')
sns.lineplot(data=yearly_trends, x='release_year', y='acousticness', ax=ax2, color='tab:orange', label='Acousticness', marker='^')
ax2.tick_params(axis='y', labelcolor='tab:red')

plt.title('Rolling Stones Catalog Trends: Popularity, Energy, and Acousticness (1964 - 2022)', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()


### Step 2.4 Observations on How Correlation Has Evolved:
* **The Acoustic Era (1960s–1970s):** In the 1960s ($+0.150$) and 1970s ($+0.184$), `acousticness` was **positively correlated** with popularity. Classic folk-blues and melodic arrangements were among the most popular hits. From the 1980s onward, this correlation turned negative/neutral.
* **The Rise of Danceability (1970s–2000s):** `danceability` emerged as the primary positive driver in the 1970s ($+0.214$), 1980s ($+0.239$), and 2000s ($+0.314$), corresponding with the band's groove-heavy disco-rock era (e.g., *'Miss You'*).
* **The Loudness Wars & Modern Remasters (1980s–2010s):** `loudness` shifted from negligible correlation in the 1960s ($+0.007$) to strong positive correlation in the 1980s ($+0.272$), 1990s ($+0.346$), and 2010s ($+0.354$), reflecting modern audio engineering standards and re-mastered compilation drops.
* **Persistent Dislike for Live Bootlegs:** In **every single decade**, `liveness` was consistently negatively correlated with popularity (ranging from $-0.127$ to $-0.407$), proving that casual streaming audiences consistently prefer crisp studio takes over audience recordings.


## Step 3: Dimensionality Reduction using PCA

### Why is Dimensionality Reduction significant here?
* **Mitigating the Curse of Dimensionality:** In a 9-dimensional space, data points become sparse and Euclidean distances converge (all points appear equidistant), reducing clustering effectiveness.
* **Noise Reduction & Multicollinearity:** Correlated features (such as `energy` and `loudness`) contain redundant information. PCA projects the data onto orthogonal axes that maximize variance while removing collinearity.
* **2D Visual Projection:** PCA allows us to project high-dimensional song vectors onto a 2D plane ($PC1$ and $PC2$) to inspect natural cluster boundaries.


In [ ]:
# 1. Standardize the audio features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[audio_features])

# 2. Fit PCA across all 9 components
pca = PCA(random_state=42)
X_pca = pca.fit_transform(X_scaled)

# 3. Calculate cumulative explained variance
cum_variance = np.cumsum(pca.explained_variance_ratio_)

# Plot Cumulative Explained Variance
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(cum_variance)+1), cum_variance, marker='o', linestyle='--', color='darkmagenta', linewidth=2)
plt.axhline(y=0.80, color='r', linestyle=':', label='80% Explained Variance Threshold')
plt.title('Cumulative Explained Variance by PCA Components', fontsize=14, fontweight='bold')
plt.xlabel('Number of Principal Components', fontsize=12)
plt.ylabel('Cumulative Explained Variance Ratio', fontsize=12)
plt.xticks(range(1, 10))
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

print('=== Explained Variance Ratio per Principal Component ===')
for idx, ev in enumerate(pca.explained_variance_ratio_):
    print(f'PC{idx+1}: {ev*100:.2f}% (Cumulative: {cum_variance[idx]*100:.2f}%)')


### Step 3.1 Observations:
* **First Two Components:** PC1 captures **$32.45\%$** of total variance, and PC2 captures **$17.94\%$**. Combined, the first two principal components preserve **$50.38\%$** of the dataset's total variance.
* **Threshold Selection:** To preserve over **$80\%$** of the total variance, 5 principal components are required ($81.38\%$).
* **2D Projection:** For 2D visualization of the cohorts, PC1 and PC2 provide a reliable, high-fidelity mapping.


### 3.2 Understanding Principal Components: Feature Loadings Analysis
To understand what PC1 and PC2 represent musically, we inspect their **eigenvector feature loadings**.


In [ ]:
# Create DataFrame of PCA loadings for the first 3 components
loadings_df = pd.DataFrame(pca.components_[:3], columns=audio_features, index=['PC1', 'PC2', 'PC3'])

# Plotting loadings heatmap
plt.figure(figsize=(12, 4))
sns.heatmap(loadings_df, annot=True, cmap='PiYG', fmt='.3f', linewidths=0.5, vmin=-0.6, vmax=0.6)
plt.title('PCA Component Loadings (Contribution of Audio Features)', fontsize=14, fontweight='bold')
plt.ylabel('Principal Component', fontsize=12)
plt.tight_layout()
plt.show()

# 2D Scatter plot of songs projected onto PC1 and PC2
plt.figure(figsize=(10, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.45, color='royalblue', edgecolors='none', s=40)
plt.title('2D Projection of Rolling Stones Songs via PC1 and PC2', fontsize=14, fontweight='bold')
plt.xlabel('Principal Component 1 (Live Energy vs Studio Polish)', fontsize=12)
plt.ylabel('Principal Component 2 (Valence & Positivity vs Acoustic Mellow)', fontsize=12)
plt.tight_layout()
plt.show()


### Step 3.2 Musical Interpretation of Principal Components:
* **PC1 (Raw Live Energy vs. Studio Polish):**
  * Strong positive loadings: `energy` ($+0.452$), `liveness` ($+0.444$), and `loudness` ($+0.370$).
  * Strong negative loading: `danceability` ($-0.418$).
  * *Meaning:* Tracks with high PC1 are fast, intense live concert performances; tracks with low PC1 are controlled, rhythmic studio tracks.
* **PC2 (Musical Positivity vs. Mellow Acoustic):**
  * Strong positive loadings: `valence` ($+0.598$), `energy` ($+0.386$), and `loudness` ($+0.357$).
  * Strong negative loading: `acousticness` ($-0.448$).
  * *Meaning:* Tracks with high PC2 are bright, cheerful, electric rock anthems; tracks with low PC2 are soft, reflective acoustic ballads.
* **PC3 (Instrumental Complexity):**
  * Dominated by `instrumentalness` ($+0.835$), isolating wordless guitar jams and extended instrumental passages.


## Step 4: Cluster Analysis

### 4.1 Finding the Optimal Number of Clusters ($K$)
We evaluate the optimal number of song cohorts using:
1. **Elbow Method (WCSS / Inertia):** Measures compactness within clusters. We look for an "elbow" point where adding more clusters yields diminishing returns.
2. **Silhouette Analysis:** Measures how distinct each cluster is from neighboring clusters. Values range from $-1$ to $+1$, with higher values indicating superior separation.


In [ ]:
wcss = []
sil_scores = []
k_range = range(2, 10)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    wcss.append(kmeans.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

# Plot Elbow and Silhouette curves side-by-side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Elbow plot
ax1.plot(k_range, wcss, marker='o', color='tab:blue', linewidth=2)
ax1.set_title('Elbow Method (Within-Cluster Sum of Squares)', fontsize=13, fontweight='bold')
ax1.set_xlabel('Number of Clusters (K)', fontsize=11)
ax1.set_ylabel('Inertia (WCSS)', fontsize=11)
ax1.axvline(x=3, color='r', linestyle='--', alpha=0.7, label='Optimal Elbow (K=3)')
ax1.legend()

# Silhouette plot
ax2.plot(k_range, sil_scores, marker='s', color='tab:orange', linewidth=2)
ax2.set_title('Silhouette Coefficient vs. Number of Clusters (K)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Number of Clusters (K)', fontsize=11)
ax2.set_ylabel('Average Silhouette Score', fontsize=11)
ax2.axvline(x=3, color='r', linestyle='--', alpha=0.7, label='Selected K=3')
ax2.legend()

plt.tight_layout()
plt.show()

print('=== Clustering Metrics Summary ===')
for k, w, s in zip(k_range, wcss, sil_scores):
    print(f'K={k}: Inertia (WCSS) = {w:.2f} | Silhouette Score = {s:.4f}')


### Step 4.1 Observations:
* **Elbow Curve:** The rate of decrease in WCSS slows down distinctly around **$K=3$**. Beyond $K=3$, the marginal gain in cluster compactness diminishes.
* **Silhouette Analysis:** The silhouette score reaches $0.2149$ at $K=2$ and remains strong at **$0.1896$ at $K=3$**. While $K=2$ produces mathematically binary clusters (e.g. live vs studio), **$K=3$ provides the optimal balance between statistical separation and rich musical interpretation** by distinguishing live tracks, upbeat studio rock, and acoustic ballads.
* **Selection:** We choose **$K=3$** as the optimal number of cohorts.


### 4.2 Training K-Means and Visualizing Cohorts with Centroids in PCA Space
We fit the final K-Means model ($K=3$), project the cluster labels onto the 2D PCA plane, and mark the **cluster centroids**.


In [ ]:
# Fit K-Means with optimal K=3
optimal_k = 3
kmeans_model = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df['cluster'] = kmeans_model.fit_predict(X_scaled)

# Project cluster centroids onto the 2D PCA space
centroids_pca = pca.transform(kmeans_model.cluster_centers_)

# Plot cohorts in 2D PCA Space
plt.figure(figsize=(11, 8))
palette = ['#e74c3c', '#2ecc71', '#3498db']
for c in range(optimal_k):
    cluster_points = X_pca[df['cluster'] == c]
    plt.scatter(cluster_points[:, 0], cluster_points[:, 1], s=45, alpha=0.55, 
                color=palette[c], label=f'Cohort {c}', edgecolors='none')

# Overlay cluster centroids
plt.scatter(centroids_pca[:, 0], centroids_pca[:, 1], s=300, c='yellow', marker='*', 
            edgecolors='black', linewidth=1.5, label='Cohort Centroids', zorder=10)

for c in range(optimal_k):
    plt.text(centroids_pca[c, 0] + 0.15, centroids_pca[c, 1] + 0.15, f'Centroid {c}', 
             fontsize=12, fontweight='bold', bbox=dict(facecolor='white', alpha=0.75, edgecolor='black', boxstyle='round,pad=0.2'))

plt.title('Song Cohorts Visualized in 2D PCA Space with Cluster Centroids (K=3)', fontsize=14, fontweight='bold')
plt.xlabel('Principal Component 1 (Live Concert Energy vs Studio Polish)', fontsize=12)
plt.ylabel('Principal Component 2 (Valence & Positivity vs Acoustic Mellow)', fontsize=12)
plt.legend(fontsize=11, loc='upper left')
plt.tight_layout()
plt.show()


## Step 5: Cohort Profiling & Music Explanations
We compute the musical profile of each cohort by calculating the unscaled feature means, standardized feature differences, and identifying iconic song representatives.


In [ ]:
# 1. Unscaled Feature Averages
cohort_profiles = df.groupby('cluster')[audio_features].mean()
print('=== Cohort Feature Averages (Original Units) ===')
display(cohort_profiles.round(3))

# 2. Standardized Feature Comparison (Z-score deviations from catalog mean)
scaler_means = pd.DataFrame(scaler.transform(cohort_profiles), columns=audio_features, index=[f'Cohort {i}' for i in range(optimal_k)])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

# Heatmap of unscaled feature means
sns.heatmap(cohort_profiles, annot=True, cmap='YlGnBu', fmt='.3f', linewidths=0.5, ax=ax1)
ax1.set_title('Cohort Audio Feature Means (Original Units)', fontsize=13, fontweight='bold')
ax1.set_ylabel('Cohort ID')

# Heatmap of standardized deviations
sns.heatmap(scaler_means, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5, ax=ax2, vmin=-1.5, vmax=1.5)
ax2.set_title('Cohort Feature Deviations from Catalog Mean (Z-Scores)', fontsize=13, fontweight='bold')
ax2.set_ylabel('Cohort')

plt.tight_layout()
plt.show()

# 3. Map human-readable cohort names
cohort_names = {
    0: 'High-Energy Live & Concert Recordings',
    1: 'Upbeat & Danceable Studio Rock Hits',
    2: 'Acoustic, Melodic & Slow Ballads'
}
df['cohort_name'] = df['cluster'].map(cohort_names)

print('\n=== Song Counts per Cohort ===')
for c, name in cohort_names.items():
    count = (df['cluster'] == c).sum()
    pct = count / len(df) * 100
    print(f'Cohort {c} ({name}): {count} songs ({pct:.1f}%)')

print('\n=== Top 3 Popular Songs in Each Cohort ===')
for c, name in cohort_names.items():
    print(f'\n--- Cohort {c}: {name} ---')
    top_songs = df[df['cluster'] == c].sort_values('popularity', ascending=False)[['name', 'album', 'popularity']].head(3)
    display(top_songs)


### Step 5 Cohort Descriptions & Profile Definitions:

Based on the feature means and standardized deviations, we define our three cohorts:

1. **Cohort 0: High-Energy Live & Concert Recordings (596 songs / 37.0%)**
   * *Sonic Profile:* Exceptionally high `liveness` ($0.821$, $+1.61\sigma$), maximum `energy` ($0.924$, $+0.78\sigma$), high `loudness` ($-5.38$ dB), and rapid `tempo` ($137.9$ BPM).
   * *Musical Characteristics:* Live stadium concerts, audience-filled bootlegs, and energetic jam versions capturing the raw intensity of the band's stage presence.
   * *Representative Tracks:* *Loving Cup (Live)*, *The Last Time (Mono Version)*, *Sway (Live / 2009 Mix)*.
   * *Recommendation Use Case:* Recommended to rock purists who enjoy the electrifying energy of live stadium performances.

2. **Cohort 1: Upbeat & Danceable Studio Rock Hits (584 songs / 36.3%)**
   * *Sonic Profile:* High `danceability` ($0.564$, $+0.59\sigma$), high `energy` ($0.821$), exceptionally high `valence` ($0.789$, $+0.98\sigma$ — cheerful/euphoric mood), and low `acousticness` ($0.186$).
   * *Musical Characteristics:* Driving studio rock anthems, funk-rock grooves, and high-tempo danceable tracks.
   * *Representative Tracks:* *Paint It, Black*, *(I Can't Get No) Satisfaction*, *Start Me Up*.
   * *Recommendation Use Case:* Ideal for workout playlists, party mixes, and road trips.

3. **Cohort 2: Acoustic, Melodic & Slow Ballads (430 songs / 26.7%)**
   * *Sonic Profile:* High `acousticness` ($0.430$, $+1.18\sigma$), lower `loudness` ($-9.72$ dB), moderate `energy` ($0.571$), and relaxed `tempo` ($115.1$ BPM).
   * *Musical Characteristics:* Soft, introspective, acoustic guitar-led ballads and soulful blues tracks focusing on emotive vocals.
   * *Representative Tracks:* *Angie*, *Wild Horses*, *Gimme Shelter*, *Sympathy For The Devil*.
   * *Recommendation Use Case:* Perfect for chill, late-night acoustic listening, and relaxation sessions.


## Step 6: Cohort-Based Song Recommendation Engine (Lesson 07 Integration)

### Connecting Cluster Cohorts to Recommendation Systems:
In **Lesson 07: Recommendation Systems**, we studied how modern streaming platforms (like Spotify, Netflix, and YouTube) implement personalized recommenders:
1. **Candidate Generation (Filtering Stage):** Rather than comparing a user's chosen song against millions of tracks, the system uses the song's **cohort (cluster)** to instantly filter the search space to musically similar tracks.
2. **Scoring & Ranking Stage (Content-Based Filtering):** Within the retrieved cohort, the system computes the **Cosine Similarity** between the target song's feature vector $\mathbf{u}$ and candidate song vectors $\mathbf{v}$:
   $$\text{Cosine Similarity}(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|_2 \|\mathbf{v}\|_2}$$
3. **Catalog Deduplication:** Music catalogs contain duplicate titles across Remasters, Deluxe Editions, and Live albums. Our recommendation engine deduplicates track titles so the listener receives a diverse playlist rather than multiple album versions of the same song.
4. **Solving the Cold-Start Problem:** Because this system relies on audio characteristics (content features) rather than historical user ratings, any newly uploaded track can be immediately clustered and recommended!


In [ ]:
def recommend_songs_from_cohort(song_title, top_n=5, deduplicate_titles=True):
    """
    Recommends top_n similar songs from the same musical cohort using Cosine Similarity.
    
    Parameters:
        song_title (str): Query song title (partial match supported).
        top_n (int): Number of recommended songs to return.
        deduplicate_titles (bool): If True, filters out alternate releases of the same song title.
    """
    # Search for matching songs
    matches = df[df['name'].str.contains(song_title, case=False, na=False)]
    if matches.empty:
        print(f"Error: No song matching '{song_title}' was found in the dataset.")
        return None
    
    # Pick the most popular release as the reference query track
    query_song = matches.sort_values('popularity', ascending=False).iloc[0]
    query_idx = query_song.name
    query_cluster = query_song['cluster']
    query_cohort = query_song['cohort_name']
    
    # Filter candidates from the SAME cohort (excluding the query song itself)
    candidates = df[(df['cluster'] == query_cluster) & (df.index != query_idx)].copy()
    
    # Compute Cosine Similarity using standardized audio features
    cand_scaled = X_scaled[candidates.index]
    q_scaled = X_scaled[query_idx].reshape(1, -1)
    similarities = cosine_similarity(q_scaled, cand_scaled).flatten()
    candidates['similarity_score'] = similarities
    
    if deduplicate_titles:
        # Extract base song title by stripping ' - Live', ' - 2009 Mix', etc.
        candidates['base_title'] = candidates['name'].str.split(' - ').str[0].str.strip()
        query_base = query_song['name'].split(' - ')[0].strip()
        candidates = candidates[candidates['base_title'].str.lower() != query_base.lower()]
        candidates = candidates.sort_values('similarity_score', ascending=False).drop_duplicates(subset=['base_title'])
    else:
        candidates = candidates.sort_values('similarity_score', ascending=False)
        
    top_recs = candidates.head(top_n)[['name', 'album', 'popularity', 'similarity_score']]
    
    print('='*75)
    print(f'QUERY SONG: "{query_song["name"]}"')
    print(f'Album: {query_song["album"]} | Popularity: {query_song["popularity"]}')
    print(f'ASSIGNED COHORT: Cohort {query_cluster} ({query_cohort})')
    print('='*75)
    print(f'Top {top_n} Musically Similar Recommendations within Cohort:')
    top_recs['similarity_score'] = top_recs['similarity_score'].round(4)
    display(top_recs)
    return top_recs

# -------------------------------------------------------------
# Test Case 1: Acoustic Melodic Ballad ("Angie") -> Expected: Cohort 2
# -------------------------------------------------------------
rec1 = recommend_songs_from_cohort('Angie', top_n=5)

# -------------------------------------------------------------
# Test Case 2: Upbeat Danceable Rock Anthem ("Satisfaction") -> Expected: Cohort 1
# -------------------------------------------------------------
rec2 = recommend_songs_from_cohort('Satisfaction', top_n=5)

# -------------------------------------------------------------
# Test Case 3: High-Energy Live Concert Track ("Street Fighting Man - Live") -> Expected: Cohort 0
# -------------------------------------------------------------
rec3 = recommend_songs_from_cohort('Street Fighting Man - Live', top_n=5)


### Step 6 Observations & Real-World System Architecture:
* **Recommendation Accuracy:**
  1. For *'Angie'* (Cohort 2: Acoustic Ballads), the engine successfully recommended *'Till The Next Goodbye'*, *'Fool To Cry'*, and *'Wild Horses'* with similarity scores $> 0.95$. All recommendations share soft acoustic guitars, slower tempos, and emotive vocals.
  2. For *'Satisfaction'* (Cohort 1: Upbeat Rock Anthems), the engine recommended high-energy, danceable rock anthems like *'Rock And A Hard Place'*, *'Terrifying'*, and *'Street Fighting Man'*.
  3. For *'Street Fighting Man - Live'* (Cohort 0: Live Concert Tracks), the engine recommended live concert recordings from *Licked Live In NYC* and *Live At The El Mocambo*.
* **Two-Stage Architecture in Production:**
  * **Stage 1 (Retrieval):** The K-Means clustering algorithm narrows millions of songs down to a single relevant cohort in $\mathcal{O}(1)$ time using the nearest cluster centroid.
  * **Stage 2 (Ranking):** Content-based Cosine Similarity ranks the top candidates within the cohort in $\mathcal{O}(M)$ time, where $M \ll N$ (total catalog size).
* **Hybrid Recommender Expansion:** In a full commercial system, this content-based cohort engine can be combined with Collaborative Filtering (matrix factorization / ALS) to rerank cohort tracks based on a specific user's listening history!
